In [1]:
!pip install -q transformers datasets accelerate

In [2]:
import os
import torch
import pandas as pd

from datasets import Dataset
from transformers import (
    GPT2LMHeadModel,
    GPT2Tokenizer,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling
)

In [3]:
dialogues = [
    "Alex: What is artificial intelligence?",
    "Sam: Artificial intelligence is the ability of machines to perform tasks that normally require human intelligence.",
    "Alex: Can AI learn from data?",
    "Sam: Yes, machine learning allows AI systems to learn patterns from data.",
    "Alex: What is machine learning?",
    "Sam: Machine learning is a method where computers learn from examples instead of being explicitly programmed.",
    "Alex: Is deep learning different from machine learning?",
    "Sam: Deep learning is a branch of machine learning that uses neural networks with multiple layers.",
    "Alex: Why are neural networks useful?",
    "Sam: Neural networks can learn complex patterns from large amounts of data.",
    "Alex: What is a dataset?",
    "Sam: A dataset is a collection of information used to train or evaluate a machine learning model.",
    "Alex: Why is data cleaning important?",
    "Sam: Clean data helps models learn useful patterns without unnecessary noise.",
    "Alex: Can AI recognize images?",
    "Sam: Yes, computer vision models can identify objects, faces, and patterns in images.",
    "Alex: What is natural language processing?",
    "Sam: Natural language processing allows computers to understand and generate human language.",
    "Alex: How does a chatbot generate answers?",
    "Sam: A language model predicts likely sequences of words based on patterns learned during training.",
    "Alex: What is GPT?",
    "Sam: GPT is a family of generative language models based on the transformer architecture.",
    "Alex: What is a transformer?",
    "Sam: A transformer is a neural network architecture that uses attention to process relationships between tokens.",
    "Alex: What does attention mean?",
    "Sam: Attention allows a model to determine which parts of the input are important when producing an output.",
    "Alex: Can AI write code?",
    "Sam: Yes, modern language models can generate, explain, and modify computer programs.",
    "Alex: Can AI completely replace programmers?",
    "Sam: AI can automate many programming tasks, but human reasoning and software design remain important.",
    "Alex: Why is human supervision useful?",
    "Sam: Human supervision helps verify whether an AI system is producing reliable and appropriate results.",
    "Alex: What is model training?",
    "Sam: Training is the process of adjusting model parameters so the model performs better on its learning objective.",
    "Alex: What is fine tuning?",
    "Sam: Fine tuning adapts a pretrained model to a specific dataset or task.",
    "Alex: Why fine tune GPT-2?",
    "Sam: Fine tuning can make GPT-2 produce text that follows a particular style or domain.",
    "Alex: Does fine tuning create a new model from scratch?",
    "Sam: No, it starts with a pretrained model and continues training it on new data.",
    "Alex: What is GPT-2?",
    "Sam: GPT-2 is a transformer-based language model designed for text generation.",
    "Alex: Is GPT-2 large?",
    "Sam: The original GPT-2 family contains models of different sizes, and the small version is convenient for experiments.",
    "Alex: What is a tokenizer?",
    "Sam: A tokenizer converts text into tokens that a language model can process.",
    "Alex: Why do models use tokens?",
    "Sam: Tokens provide a numerical representation of language that neural networks can process.",
    "Alex: What happens after tokenization?",
    "Sam: The token IDs are passed into the neural network to predict the next tokens.",
    "Alex: What is next-token prediction?",
    "Sam: It is the task of predicting the most likely following token given previous tokens.",
    "Alex: Is GPT-2 trained using next-token prediction?",
    "Sam: Yes, GPT-2 learns to predict the next token in sequences of text.",
    "Alex: What is a language model?",
    "Sam: A language model estimates the probability of sequences of tokens.",
    "Alex: Can a language model understand everything?",
    "Sam: No, it learns statistical patterns and does not have human understanding in the same way people do.",
    "Alex: Why does training data matter?",
    "Sam: Training data strongly influences the patterns and information a model learns.",
    "Alex: What happens if the dataset is poor?",
    "Sam: Poor data can lead to unreliable or biased model behavior.",
    "Alex: What is overfitting?",
    "Sam: Overfitting happens when a model learns the training examples too closely and performs poorly on new data.",
    "Alex: How can overfitting be reduced?",
    "Sam: More diverse data, regularization, validation, and appropriate training settings can help.",
    "Alex: What is an epoch?",
    "Sam: An epoch represents one complete pass through the training dataset.",
    "Alex: Should we always train for many epochs?",
    "Sam: No, too many epochs can cause overfitting, especially with a small dataset.",
    "Alex: What is a batch?",
    "Sam: A batch is a group of training examples processed together.",
    "Alex: What is the learning rate?",
    "Sam: The learning rate controls how strongly model parameters are updated during training.",
    "Alex: What happens if the learning rate is too high?",
    "Sam: Training can become unstable and the model may fail to learn properly.",
    "Alex: What happens if it is too low?",
    "Sam: Training may become very slow.",
    "Alex: What is a loss function?",
    "Sam: A loss function measures how different the model prediction is from the expected target.",
    "Alex: Does lower loss always mean a better model?",
    "Sam: Lower training loss is useful, but validation performance is also important.",
    "Alex: What is the Hugging Face library?",
    "Sam: Hugging Face provides tools and pretrained models for natural language processing and machine learning.",
    "Alex: What is the Transformers library?",
    "Sam: Transformers provides implementations of many transformer-based models and training utilities.",
    "Alex: What is Trainer?",
    "Sam: Trainer is a Hugging Face class that simplifies the training and evaluation process.",
    "Alex: Why use Trainer?",
    "Sam: It handles many training details such as optimization, logging, saving checkpoints, and evaluation.",
    "Alex: Can GPT-2 generate text after fine tuning?",
    "Sam: Yes, the fine-tuned model can generate text influenced by the new training style.",
    "Alex: Will the output always be correct?",
    "Sam: No, generated text can still contain errors or unexpected statements.",
    "Alex: What is temperature?",
    "Sam: Temperature controls the randomness of token selection during text generation.",
    "Alex: What happens with a low temperature?",
    "Sam: A low temperature generally produces more predictable and conservative text.",
    "Alex: What happens with a high temperature?",
    "Sam: A high temperature generally produces more varied and creative text.",
    "Alex: What is top-k sampling?",
    "Sam: Top-k sampling restricts token selection to the k most probable tokens.",
    "Alex: What is top-p sampling?",
    "Sam: Top-p sampling selects from the smallest group of tokens whose cumulative probability reaches a chosen value.",
    "Alex: Why are sampling methods useful?",
    "Sam: They help control the diversity and randomness of generated text.",
    "Alex: Can fine tuning change vocabulary?",
    "Sam: It can increase the model's tendency to use vocabulary and patterns present in the new dataset.",
    "Alex: Can fine tuning change sentence structure?",
    "Sam: Yes, repeated patterns in the dataset can influence sentence structure and response style.",
    "Alex: Can fine tuning change tone?",
    "Sam: Yes, consistent training examples can encourage a particular tone.",
    "Alex: Why is dataset consistency important?",
    "Sam: Consistent examples make it easier for the model to learn the desired writing pattern.",
    "Alex: What happens if the dataset contains unrelated text?",
    "Sam: The model may learn mixed styles and produce less consistent outputs.",
    "Alex: Should timestamps be removed?",
    "Sam: Yes, unnecessary timestamps should be removed during dataset cleaning.",
    "Alex: Should usernames be removed?",
    "Sam: Yes, usernames and other unnecessary metadata should normally be removed.",
    "Alex: Why clean the dataset?",
    "Sam: Cleaning removes noise and leaves the language patterns that we actually want the model to learn.",
    "Alex: What is text generation?",
    "Sam: Text generation is the process of producing new text from a language model.",
    "Alex: Can we compare two language models?",
    "Sam: Yes, we can provide the same prompt to both models and compare their generated outputs.",
    "Alex: What should we compare?",
    "Sam: We can compare vocabulary, sentence structure, tone, consistency, and topic relevance.",
    "Alex: What would a successful fine tuning result look like?",
    "Sam: The generated text should show noticeable characteristics of the custom training style.",
    "Alex: Does fine tuning guarantee improvement?",
    "Sam: No, the result depends on the quality and size of the dataset and the training settings.",
    "Alex: Why use a small GPT-2 model?",
    "Sam: The small model requires fewer computational resources and is suitable for educational experiments.",
    "Alex: Can we run GPT-2 on Google Colab?",
    "Sam: Yes, Google Colab provides a convenient environment for running transformer models.",
    "Alex: Is GPU useful?",
    "Sam: Yes, a GPU can significantly speed up neural network training.",
    "Alex: How can we check GPU availability?",
    "Sam: We can use PyTorch to check whether CUDA is available.",
    "Alex: What is CUDA?",
    "Sam: CUDA is a platform that allows software to use compatible NVIDIA GPUs for computation.",
    "Alex: What is a checkpoint?",
    "Sam: A checkpoint is a saved version of a model during or after training.",
    "Alex: Why save checkpoints?",
    "Sam: Checkpoints allow us to continue training or reuse a trained model later.",
    "Alex: Can we save the final model?",
    "Sam: Yes, the trained model and tokenizer can be saved to a directory.",
    "Alex: Can we download the model from Colab?",
    "Sam: Yes, the saved model folder can be compressed and downloaded.",
    "Alex: What is inference?",
    "Sam: Inference is the process of using a trained model to produce predictions or generated text.",
    "Alex: Is inference the same as training?",
    "Sam: No, training updates model parameters while inference uses the existing parameters.",
    "Alex: Why use the same prompt for comparison?",
    "Sam: The same prompt makes the comparison more meaningful because both models receive the same input.",
    "Alex: What style does our dataset use?",
    "Sam: Our dataset uses a structured educational conversation between two speakers.",
    "Alex: How can the fine-tuned model learn this style?",
    "Sam: By repeatedly seeing the dialogue structure during training.",
    "Alex: What pattern appears in our dataset?",
    "Sam: The dialogue follows alternating questions from Alex and informative responses from Sam.",
    "Alex: Will the fine-tuned model always alternate perfectly?",
    "Sam: Not necessarily, but the pattern should become more likely after fine tuning.",
    "Alex: Can a model learn short answers?",
    "Sam: Yes, the length and structure of training examples can influence generated responses.",
    "Alex: Can a model learn technical vocabulary?",
    "Sam: Yes, repeated technical terms can become more common in generated text.",
    "Alex: What technical terms are present here?",
    "Sam: Terms such as transformer, tokenizer, dataset, training, inference, sampling, and fine tuning are common.",
    "Alex: Why repeat technical concepts?",
    "Sam: Repetition provides the model with stronger examples of the desired domain and style.",
    "Alex: What is domain adaptation?",
    "Sam: Domain adaptation is the process of adjusting a model to perform better in a particular subject area.",
    "Alex: Is fine tuning a form of domain adaptation?",
    "Sam: Fine tuning can be used for domain adaptation when the training data represents a specific domain.",
    "Alex: Can we fine tune on conversations?",
    "Sam: Yes, conversational datasets are commonly used for language model adaptation.",
    "Alex: Can we fine tune on stories?",
    "Sam: Yes, a story dataset can encourage the model to generate narrative-style text.",
    "Alex: Can we fine tune on code?",
    "Sam: Yes, code datasets can make a model more likely to generate programming-related patterns.",
    "Alex: What is the main goal of this experiment?",
    "Sam: The main goal is to observe how a pretrained GPT-2 model changes after learning a custom writing style.",
    "Alex: What is the base model?",
    "Sam: The base model is GPT-2 before any additional training on our custom dataset.",
    "Alex: What is the fine-tuned model?",
    "Sam: The fine-tuned model is GPT-2 after additional training on our custom dialogue dataset.",
    "Alex: What do we expect from the base model?",
    "Sam: The base model should produce more general text because it has not learned our specific dialogue pattern.",
    "Alex: What do we expect from the fine-tuned model?",
    "Sam: We expect it to produce more dialogue-like text using technical vocabulary and question-answer patterns.",
    "Alex: How do we evaluate the result?",
    "Sam: We compare outputs from both models using the same prompt.",
    "Alex: What is one possible difference?",
    "Sam: The fine-tuned model may use more technical vocabulary.",
    "Alex: What is another possible difference?",
    "Sam: The fine-tuned model may follow the question-and-answer format more consistently.",
    "Alex: Can fine tuning make the model more focused?",
    "Sam: Yes, when the dataset is narrow and consistent, generated text can become more domain focused.",
    "Alex: Can it also make the model repetitive?",
    "Sam: Yes, a small dataset may cause the model to repeat patterns seen frequently during training.",
    "Alex: Why is dataset size important?",
    "Sam: A larger and diverse dataset can provide more examples of the target style.",
    "Alex: Is 200 to 300 lines enough for this assignment?",
    "Sam: It is enough for a small educational fine tuning experiment.",
    "Alex: Should we expect professional-level results?",
    "Sam: No, the purpose is to demonstrate style adaptation rather than production-level performance.",
    "Alex: What should we document?",
    "Sam: We should document the dataset, training process, prompts, outputs, and differences between the models.",
    "Alex: Why compare before and after?",
    "Sam: The comparison demonstrates whether fine tuning actually changed the model's behavior.",
    "Alex: What should the final report contain?",
    "Sam: It should contain the objective, dataset preparation, methodology, results, comparison, and conclusion.",
    "Alex: What is the conclusion we expect?",
    "Sam: We expect fine tuning to make GPT-2 more consistent with the style and vocabulary of the custom dataset.",
    "Alex: Is this experiment useful for understanding transfer learning?",
    "Sam: Yes, it demonstrates how pretrained knowledge can be adapted to a specialized dataset.",
    "Alex: Can the same idea be used in real applications?",
    "Sam: Yes, organizations can adapt models to specialized domains, although larger datasets and careful evaluation are normally required.",
    "Alex: What is the most important lesson?",
    "Sam: A pretrained language model can be adapted to a new writing style through additional training.",
    "Alex: So fine tuning changes the behavior of a pretrained model?",
    "Sam: Exactly. Fine tuning adjusts the model so its generated text becomes more aligned with the examples it receives.",
]

print("Number of lines:", len(dialogues))

Number of lines: 224


In [4]:
with open("custom_dialogue_dataset.txt", "w", encoding="utf-8") as f:
    for line in dialogues:
        f.write(line + "\n")

print("Dataset saved successfully.")

Dataset saved successfully.


In [5]:
cleaned_lines = []

for line in dialogues:
    line = line.strip()

    # Remove empty lines
    if not line:
        continue

    # Remove unwanted formatting if present
    line = line.replace("\t", " ")
    line = " ".join(line.split())

    cleaned_lines.append(line)

print("Original lines:", len(dialogues))
print("Cleaned lines:", len(cleaned_lines))

Original lines: 224
Cleaned lines: 224


In [6]:
dataset = Dataset.from_dict({
    "text": cleaned_lines
})

dataset

Dataset({
    features: ['text'],
    num_rows: 224
})

In [7]:
model_name = "gpt2"

tokenizer = GPT2Tokenizer.from_pretrained(model_name)

# GPT-2 does not have a pad token by default
tokenizer.pad_token = tokenizer.eos_token

print("Tokenizer loaded successfully.")

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Tokenizer loaded successfully.


In [8]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=128
    )

tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

print("Tokenization completed.")

Map:   0%|          | 0/224 [00:00<?, ? examples/s]

Tokenization completed.


In [9]:
base_model = GPT2LMHeadModel.from_pretrained(model_name)

base_model.config.pad_token_id = tokenizer.pad_token_id

print("Base GPT-2 loaded.")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Base GPT-2 loaded.


In [10]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./gpt2-finetuned",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    learning_rate=5e-5,
    weight_decay=0.01,
    logging_steps=10,
    save_strategy="epoch",
    report_to="none",
    fp16=torch.cuda.is_available()
)

print("Training arguments created successfully!")


Training arguments created successfully!


In [11]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

In [12]:
trainer = Trainer(
    model=base_model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)

print("Trainer created.")

Trainer created.


In [13]:
trainer.train()

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
10,4.566685
20,3.821444
30,3.313956
40,2.993177
50,2.717077
60,3.216773
70,2.479576
80,2.502733
90,2.296579
100,2.302550


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=168, training_loss=2.6666517484755743, metrics={'train_runtime': 71.3649, 'train_samples_per_second': 9.416, 'train_steps_per_second': 2.354, 'total_flos': 5401396224000.0, 'train_loss': 2.6666517484755743, 'epoch': 3.0})

In [14]:
trainer.save_model("./gpt2-finetuned")
tokenizer.save_pretrained("./gpt2-finetuned")

print("Fine-tuned model saved successfully.")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Fine-tuned model saved successfully.


In [15]:
fine_tuned_model = GPT2LMHeadModel.from_pretrained(
    "./gpt2-finetuned"
)

fine_tuned_model.eval()

print("Fine-tuned model loaded.")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Fine-tuned model loaded.


In [16]:
prompt = "Artificial intelligence is"

In [17]:
def generate_text(model, prompt, max_length=120):
    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    )

    inputs = {
        key: value.to(model.device)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_length=max_length,
            do_sample=True,
            temperature=0.8,
            top_k=50,
            top_p=0.95,
            no_repeat_ngram_size=2,
            pad_token_id=tokenizer.eos_token_id
        )

    return tokenizer.decode(
        output[0],
        skip_special_tokens=True
    )

In [18]:
base_output = generate_text(
    base_model,
    prompt
)

print("===== ORIGINAL GPT-2 OUTPUT =====")
print(base_output)

===== ORIGINAL GPT-2 OUTPUT =====
Artificial intelligence is the ability of computers to process information. A computer can automate many tasks, from programming a software program to preparing text for use on a website. Human intelligence can recognize and learn new vocabulary from natural language processing. An AI system learns how to fine tune a human language model. AI systems can also predict human-style responses. The goal of AI training is to train AI models to learn how human speakers use language. Training algorithms use the same techniques used in AI to recognize human sentences. Neural networks can train models that learn multiple information from different parts of the dataset. Machine learning models


In [19]:
fine_tuned_output = generate_text(
    fine_tuned_model,
    prompt
)

print("===== FINE-TUNED GPT-2 OUTPUT =====")
print(fine_tuned_output)

===== FINE-TUNED GPT-2 OUTPUT =====
Artificial intelligence is the ability of computers to process data from human input. It uses artificial intelligence to perform tasks such as learning, reasoning, and inference.

What is AI? AI is a method of AI that uses natural language processing to train and process computer programs. AI allows machines to learn from examples and make decisions. In this talk, Artificial Intelligence is covered. Machine learning uses machine learning to improve machine intelligence.


In [20]:
print("=" * 70)
print("BEFORE AND AFTER COMPARISON")
print("=" * 70)

print("\nORIGINAL GPT-2:\n")
print(base_output)

print("\n" + "=" * 70)

print("\nFINE-TUNED GPT-2:\n")
print(fine_tuned_output)

BEFORE AND AFTER COMPARISON

ORIGINAL GPT-2:

Artificial intelligence is the ability of computers to process information. A computer can automate many tasks, from programming a software program to preparing text for use on a website. Human intelligence can recognize and learn new vocabulary from natural language processing. An AI system learns how to fine tune a human language model. AI systems can also predict human-style responses. The goal of AI training is to train AI models to learn how human speakers use language. Training algorithms use the same techniques used in AI to recognize human sentences. Neural networks can train models that learn multiple information from different parts of the dataset. Machine learning models


FINE-TUNED GPT-2:

Artificial intelligence is the ability of computers to process data from human input. It uses artificial intelligence to perform tasks such as learning, reasoning, and inference.

What is AI? AI is a method of AI that uses natural language pr

In [21]:
with open("base_model_output.txt", "w", encoding="utf-8") as f:
    f.write(base_output)

with open("fine_tuned_model_output.txt", "w", encoding="utf-8") as f:
    f.write(fine_tuned_output)

print("Outputs saved.")

Outputs saved.
